# 第3章　はじめての学習ループ（線形回帰）

第2章の勾配降下を、いよいよ「データから直線を学習する」問題に使います。
ここで **PyTorch 学習の“5ステップの核”** が完成します。どんな巨大モデルでもこの骨格は同じです。

この章のゴール：データから $y = wx + b$ の $w,b$ を学習でき、`optimizer` の役割が分かる。

> **このノートの使い方**
> - 上から順にセルを実行（Colab/Jupyter ともに `Shift + Enter`）。
> - コードは**少し書き換えて壊して直す**のが一番伸びます。各章末に演習があります。
> - GPU は不要な章が多いです。重い章（CNN）では使い方を案内します。

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA (GPU) available:", torch.cuda.is_available())

## 3-1. 練習データを作る
「本当は $y = 2x + 1$」というデータに、少しノイズを乗せて用意します。これを当てるのが目標。

In [ ]:
import torch
torch.manual_seed(0)                       # 再現性のため乱数を固定

X = torch.linspace(-3, 3, 100).unsqueeze(1)   # 形 (100, 1)
true_w, true_b = 2.0, 1.0
y = true_w * X + true_b + 0.5 * torch.randn(X.shape)   # ノイズ付き

print("X:", X.shape, " y:", y.shape)

## 3-2. まず「手動」で学習（中身を完全に理解する）

- パラメータ `w, b` を `requires_grad=True` で用意。
- **予測（forward）**：`pred = w*X + b`
- **損失（loss）**：予測と正解のズレ。回帰では**平均二乗誤差 (MSE)** $\frac1N\sum (pred-y)^2$。
- **勾配（backward）** → **更新** → **リセット**。

In [ ]:
w = torch.zeros(1, requires_grad=True)   # 0 からスタート
b = torch.zeros(1, requires_grad=True)
lr = 0.05

for epoch in range(100):
    pred = w * X + b                  # ① 予測（forward）
    loss = ((pred - y) ** 2).mean()   # ② 損失（MSE）

    loss.backward()                   # ③ 勾配を計算
    with torch.no_grad():
        w -= lr * w.grad              # ④ 更新
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()    # ⑤ リセット

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.4f}  w={w.item():.3f}  b={b.item():.3f}")

print(f"学習結果  w={w.item():.3f} (正解2.0),  b={b.item():.3f} (正解1.0)")

## 3-3. `optimizer` で同じことを書く（実務スタイル）

手動の更新部分（`w -= lr*w.grad ...` と `zero_()`）は、`torch.optim` がまとめてやってくれます。
ここで登場するのが **5ステップの核**：

```
optimizer.zero_grad()   # ① 勾配リセット
pred = model(x)         # ② 予測（forward）
loss = criterion(...)   # ③ 損失
loss.backward()         # ④ 勾配
optimizer.step()        # ⑤ 更新
```

`nn.Linear(1, 1)` は「入力1次元→出力1次元の直線」、つまり内部に `w, b` を持つモデルです。

In [ ]:
import torch.nn as nn

model = nn.Linear(1, 1)                       # y = w*x + b を表すモデル
criterion = nn.MSELoss()                      # 損失関数（平均二乗誤差）
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)   # 最適化アルゴリズム

for epoch in range(100):
    optimizer.zero_grad()        # ① 勾配リセット
    pred = model(X)              # ② 予測
    loss = criterion(pred, y)    # ③ 損失
    loss.backward()              # ④ 勾配
    optimizer.step()             # ⑤ 更新

    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}: loss={loss.item():.4f}")

w_learned = model.weight.item()
b_learned = model.bias.item()
print(f"学習結果  w={w_learned:.3f} (正解2.0),  b={b_learned:.3f} (正解1.0)")

## 3-4. 結果を可視化
学習した直線が、データの真ん中を通っているか目で確認します。

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():                 # 予測だけなので勾配不要
    pred_line = model(X)

plt.figure(figsize=(6, 4))
plt.scatter(X.numpy(), y.numpy(), s=12, label="data")
plt.plot(X.numpy(), pred_line.numpy(), color="red", linewidth=2, label="learned line")
plt.legend(); plt.title("Linear Regression"); plt.xlabel("x"); plt.ylabel("y")
plt.show()

## 演習 3
1. `lr` を `0.5` や `0.001` に変えて、収束の速さ／発散を観察しよう。
2. 真の関係を `y = -3x + 2` に変えて、ちゃんと学習できるか確かめよう。
3. `SGD` を `torch.optim.Adam(model.parameters(), lr=0.1)` に変えると収束はどう変わる？
4. 損失の推移を `losses.append(loss.item())` で記録して、`plt.plot(losses)` で学習曲線を描こう。

In [ ]:
# ここに自分のコードを書いて実行してみよう
